# AgriNexus AI — Research-Grade Notebook 04: Smart Irrigation Forecasting

**Task**: Time-Series 3-Hour-Ahead Soil Water Content ($SWC_{t+3h}$) Forecasting & Agronomic Irrigation Decision Engine
**Primary Dataset**: Gallipoli Sensor Time-Series (`SoilWaterContent.xlsx` & `porta1_meteo_piogge.xlsx`)
**Scientific Focus**: Strict Temporal Causality Audit (Zero Future Leakage), Chronological Partitioning, Persistence Baseline Benchmarking ($SWC_{t+3h} = SWC_t$), Explicit Validation ML Champion vs Final Test Champion Selection, Unseen Test Set MAE Improvement Calculation, Separated Agronomic Rule-Based Decision Logic, and Artifact Reload Verification.

In [1]:
# Section 1: Environment, Dependencies & Deterministic Seed Setup
import os
import sys
import math
import time
import json
import random
import pickle
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# Deterministic Seed Setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

DATA_DIR = Path('d:/PROJECTS/AGRINEXUS-AI/data/raw/irrigation')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data/raw/irrigation')

MODELS_DIR = Path('d:/PROJECTS/AGRINEXUS-AI/Notebook/models')
if not MODELS_DIR.exists():
    MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment Ready | Seed: {SEED}")
print(f"Data Path: {DATA_DIR.resolve()}")
print(f"Models Directory: {MODELS_DIR.resolve()}")

Environment Ready | Seed: 42
Data Path: D:\PROJECTS\AGRINEXUS-AI\data\raw\irrigation
Models Directory: D:\PROJECTS\AGRINEXUS-AI\Notebook\models


## 2. Problem Statement & Strict Causality Audit
Forecasting root-zone Soil Water Content ($SWC$, in $\text{m}^3/\text{m}^3$) 3 hours ahead ($SWC_{t+3h}$) enables proactive irrigation scheduling.

### Causality & Temporal Leakage Rules:
1. Features at time $t$ MUST depend ONLY on historical observations available at or before $t$.
2. **Forbidden**: Future interpolation (`bfill`), backward rolling windows containing future observations, or future weather forecasts.
3. **Persistence Baseline**: Predicts $SWC_{t+3h} = SWC_t$. Any candidate model must be rigorously compared against persistence on held-out test data.

In [2]:
# Section 3: Time-Series Preprocessing & Historical Lag Engineering
swc_excel = DATA_DIR / "SoilWaterContent.xlsx"
meteo_excel = DATA_DIR / "porta1_meteo_piogge.xlsx"

assert swc_excel.exists(), f"Missing {swc_excel}"
assert meteo_excel.exists(), f"Missing {meteo_excel}"

df_swc = pd.read_excel(swc_excel)
df_meteo = pd.read_excel(meteo_excel)

df_swc.columns = [c.strip() for c in df_swc.columns]
df_meteo.columns = [c.strip() for c in df_meteo.columns]

df_swc['Timestamps'] = pd.to_datetime(df_swc['Timestamps'])
df_meteo['Data'] = pd.to_datetime(df_meteo['Data'])

df_swc = df_swc.sort_values('Timestamps').reset_index(drop=True)
df_meteo = df_meteo.sort_values('Data').reset_index(drop=True)

df_swc['SWC'] = (df_swc['Port1'] + df_swc['Port2']) / 2.0

# Strict Backward-Only Merge
df_merged = pd.merge_asof(df_swc, df_meteo, left_on='Timestamps', right_on='Data', direction='backward')
rain_col = [c for c in df_merged.columns if 'Pioggia' in c or 'pioggia' in c or 'rain' in c.lower()][0]
df_merged['Rainfall_mm'] = df_merged[rain_col].fillna(0.0)

# Resample to 1-Hour Regular Time Steps
df_hourly = df_merged.set_index('Timestamps').resample('1h').agg({
    'SWC': 'mean',
    'Rainfall_mm': 'sum'
}).reset_index()

df_hourly['SWC'] = df_hourly['SWC'].ffill()

# Target: 3-Hour-Ahead Soil Water Content (SWC t+3h)
HORIZON = 3
df_hourly['SWC_target_3h'] = df_hourly['SWC'].shift(-HORIZON)

# Historical Lag Features
df_hourly['SWC_lag1h'] = df_hourly['SWC'].shift(1)
df_hourly['SWC_lag2h'] = df_hourly['SWC'].shift(2)
df_hourly['SWC_lag3h'] = df_hourly['SWC'].shift(3)
df_hourly['SWC_roll6h_mean'] = df_hourly['SWC'].shift(1).rolling(6).mean()
df_hourly['Rain_roll6h_sum'] = df_hourly['Rainfall_mm'].rolling(6).sum()

df_clean = df_hourly.dropna().reset_index(drop=True)
print(f"Processed Hourly Time-Series Dataset: {len(df_clean):,} observations")
print(f"Date Range: {df_clean['Timestamps'].min()} to {df_clean['Timestamps'].max()}")

Processed Hourly Time-Series Dataset: 14,588 observations
Date Range: 2021-07-01 01:00:00 to 2023-02-28 20:00:00


In [3]:
# Section 4: Chronological Time-Series Train / Validation / Test Split
feature_cols = ['SWC', 'SWC_lag1h', 'SWC_lag2h', 'SWC_lag3h', 'SWC_roll6h_mean', 'Rainfall_mm', 'Rain_roll6h_sum']
target_col = 'SWC_target_3h'

n = len(df_clean)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

train_df = df_clean.iloc[:n_train].copy()
val_df = df_clean.iloc[n_train:n_train+n_val].copy()
test_df = df_clean.iloc[n_train+n_val:].copy()

print(f"Chronological Split Sizes:")
print(f"  - Train partition: {len(train_df):,} samples (70%) | {train_df['Timestamps'].min()} to {train_df['Timestamps'].max()}")
print(f"  - Val partition:   {len(val_df):,} samples (15%) | {val_df['Timestamps'].min()} to {val_df['Timestamps'].max()}")
print(f"  - Test partition:  {len(test_df):,} samples (15%) | {test_df['Timestamps'].min()} to {test_df['Timestamps'].max()}")

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Chronological Split Sizes:
  - Train partition: 10,211 samples (70%) | 2021-07-01 01:00:00 to 2022-08-30 11:00:00
  - Val partition:   2,188 samples (15%) | 2022-08-30 12:00:00 to 2022-11-29 15:00:00
  - Test partition:  2,189 samples (15%) | 2022-11-29 16:00:00 to 2023-02-28 20:00:00


In [4]:
# Section 5: Persistence Baseline vs ML Model Benchmarking (Validation Partition)
# Persistence Baseline: SWC_{t+3h} = SWC_t
p_val_preds = val_df['SWC'].values
p_val_mae = mean_absolute_error(y_val, p_val_preds)
p_val_rmse = math.sqrt(mean_squared_error(y_val, p_val_preds))
p_val_r2 = r2_score(y_val, p_val_preds)

print(f"Persistence Baseline (SWC_t+3h = SWC_t) Validation Performance:")
print(f"  - Persistence Val MAE:  {p_val_mae:.6f} m3/m3")
print(f"  - Persistence Val RMSE: {p_val_rmse:.6f} m3/m3")
print(f"  - Persistence Val R2:   {p_val_r2:.6f}")

ml_candidates = {
    'Ridge Regression': Ridge(alpha=10.0, random_state=SEED),
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=100, random_state=SEED)
}

benchmark_results = [{
    'Model': 'Persistence Baseline',
    'Val MAE': p_val_mae,
    'Val RMSE': p_val_rmse,
    'Val R2': p_val_r2
}]

best_val_mae = float('inf')
val_ml_champion_name = None
val_ml_champion_model = None

print("\nBenchmarking Candidate ML Regressors on Validation Partition...")
for name, model in ml_candidates.items():
    model.fit(X_train_scaled, y_train)
    val_preds = model.predict(X_val_scaled)
    
    mae = mean_absolute_error(y_val, val_preds)
    rmse = math.sqrt(mean_squared_error(y_val, val_preds))
    r2 = r2_score(y_val, val_preds)
    
    benchmark_results.append({
        'Model': name,
        'Val MAE': mae,
        'Val RMSE': rmse,
        'Val R2': r2
    })
    print(f"  {name:<22} | Val MAE: {mae:.6f} | Val RMSE: {rmse:.6f} | Val R2: {r2:.4f}")
    
    if mae < best_val_mae:
        best_val_mae = mae
        val_ml_champion_name = name
        val_ml_champion_model = model

df_bench = pd.DataFrame(benchmark_results)
print(f"\nVALIDATION ML CHAMPION: {val_ml_champion_name} (Val MAE = {best_val_mae:.6f} m3/m3)")

Persistence Baseline (SWC_t+3h = SWC_t) Validation Performance:
  - Persistence Val MAE:  0.000786 m3/m3
  - Persistence Val RMSE: 0.004745 m3/m3
  - Persistence Val R2:   0.985350

Benchmarking Candidate ML Regressors on Validation Partition...
  Ridge Regression       | Val MAE: 0.000920 | Val RMSE: 0.004077 | Val R2: 0.9892
  Linear Regression      | Val MAE: 0.000833 | Val RMSE: 0.003940 | Val R2: 0.9899


  Random Forest          | Val MAE: 0.001226 | Val RMSE: 0.004580 | Val R2: 0.9863


  Extra Trees            | Val MAE: 0.001149 | Val RMSE: 0.004666 | Val R2: 0.9858


  HistGradientBoosting   | Val MAE: 0.001324 | Val RMSE: 0.004600 | Val R2: 0.9862

VALIDATION ML CHAMPION: Linear Regression (Val MAE = 0.000833 m3/m3)


In [5]:
# Section 6: Held-Out Test Evaluation — Validation ML Champion vs Final Test Champion
p_test_preds = test_df['SWC'].values
p_test_mae = mean_absolute_error(y_test, p_test_preds)
p_test_rmse = math.sqrt(mean_squared_error(y_test, p_test_preds))
p_test_r2 = r2_score(y_test, p_test_preds)

ml_test_preds = val_ml_champion_model.predict(X_test_scaled)
ml_test_mae = mean_absolute_error(y_test, ml_test_preds)
ml_test_rmse = math.sqrt(mean_squared_error(y_test, ml_test_preds))
ml_test_r2 = r2_score(y_test, ml_test_preds)

# Improvement Formula: (Baseline - Model) / Baseline * 100
mae_improvement_pct = ((p_test_mae - ml_test_mae) / p_test_mae) * 100.0
rmse_improvement_pct = ((p_test_rmse - ml_test_rmse) / p_test_rmse) * 100.0

# Determine Final Test Champion based on test MAE
if ml_test_mae < p_test_mae:
    final_test_champion = f"{val_ml_champion_name} (ML Champion)"
    readiness_status = "PASS"
else:
    final_test_champion = "Persistence Baseline"
    readiness_status = "CONDITIONAL"

print("="*70)
print("MANDATORY FINAL TEST SET COMPARISON TABLE")
print("="*70)
test_comparison = [
    {"Model Role": "Persistence Baseline", "Model Name": "Persistence (SWC_t+3h = SWC_t)", "MAE (m3/m3)": f"{p_test_mae:.6f}", "RMSE (m3/m3)": f"{p_test_rmse:.6f}", "R2": f"{p_test_r2:.4f}", "MAE Impr (%)": "0.00%"},
    {"Model Role": "Validation ML Champion", "Model Name": f"{val_ml_champion_name}", "MAE (m3/m3)": f"{ml_test_mae:.6f}", "RMSE (m3/m3)": f"{ml_test_rmse:.6f}", "R2": f"{ml_test_r2:.4f}", "MAE Impr (%)": f"{mae_improvement_pct:+.2f}%"},
    {"Model Role": "FINAL TEST CHAMPION", "Model Name": final_test_champion, "MAE (m3/m3)": f"{min(p_test_mae, ml_test_mae):.6f}", "RMSE (m3/m3)": f"{min(p_test_rmse, ml_test_rmse):.6f}", "R2": f"{max(p_test_r2, ml_test_r2):.4f}", "MAE Impr (%)": f"{max(0.0, mae_improvement_pct):+.2f}%"}
]
print(pd.DataFrame(test_comparison).to_string(index=False))
print("="*70)

print(f"\nScientific Conclusion: Validation ML Champion = {val_ml_champion_name} | Final Test Champion = {final_test_champion}")
print(f"MAE Improvement vs Persistence: {mae_improvement_pct:+.2f}%")

MANDATORY FINAL TEST SET COMPARISON TABLE
            Model Role                     Model Name MAE (m3/m3) RMSE (m3/m3)     R2 MAE Impr (%)
  Persistence Baseline Persistence (SWC_t+3h = SWC_t)    0.000311     0.001325 0.9905        0.00%
Validation ML Champion              Linear Regression    0.000519     0.001509 0.9877      -67.01%
   FINAL TEST CHAMPION           Persistence Baseline    0.000311     0.001325 0.9905       +0.00%

Scientific Conclusion: Validation ML Champion = Linear Regression | Final Test Champion = Persistence Baseline
MAE Improvement vs Persistence: -67.01%


In [6]:
# Section 7: Agronomic Irrigation Decision Engine (Separated Architecture)
FC = 0.320       # Field Capacity (m3/m3)
WP = 0.140       # Wilting Point (m3/m3)
MAD = 0.50       # Management Allowed Depletion (50% TAW)
TAW = FC - WP    # Total Available Water = 0.180 m3/m3
RAW = MAD * TAW  # Readily Available Water = 0.090 m3/m3
CRITICAL_THRESHOLD = FC - RAW  # 0.230 m3/m3
ROOT_DEPTH_M = 0.60   # Root zone depth (meters)
FLOW_RATE_LS = 1.2    # Irrigation pump flow rate (liters / second)

def compute_irrigation_recommendation(predicted_swc_3h, current_swc):
    if predicted_swc_3h < CRITICAL_THRESHOLD:
        deficit_m3_per_m3 = FC - predicted_swc_3h
        water_depth_mm = deficit_m3_per_m3 * ROOT_DEPTH_M * 1000.0
        water_volume_liters_per_ha = water_depth_mm * 10.0 * 1000.0
        duration_hours = (water_volume_liters_per_ha / (FLOW_RATE_LS * 3600.0))
        return {
            'action': 'IRRIGATE',
            'predicted_swc_3h': round(float(predicted_swc_3h), 4),
            'threshold_swc': CRITICAL_THRESHOLD,
            'water_depth_mm': round(float(water_depth_mm), 2),
            'water_volume_l_ha': round(float(water_volume_liters_per_ha), 1),
            'recommended_duration_hours': round(float(duration_hours), 2)
        }
    else:
        return {
            'action': 'HOLD',
            'predicted_swc_3h': round(float(predicted_swc_3h), 4),
            'threshold_swc': CRITICAL_THRESHOLD,
            'water_depth_mm': 0.0,
            'water_volume_l_ha': 0.0,
            'recommended_duration_hours': 0.0
        }

sample_pred = float(ml_test_preds[0])
sample_curr = float(X_test.iloc[0]['SWC'])
decision = compute_irrigation_recommendation(sample_pred, sample_curr)
print("Agronomic Decision Engine Test Output:")
print(json.dumps(decision, indent=2))

Agronomic Decision Engine Test Output:
{
  "action": "HOLD",
  "predicted_swc_3h": 0.2496,
  "threshold_swc": 0.23,
  "water_depth_mm": 0.0,
  "water_volume_l_ha": 0.0,
  "recommended_duration_hours": 0.0
}


In [7]:
# Section 8: Model Artifact Serialization & Reload Verification
artifact_filename = "irrigation_prediction.pkl"
artifact_path = MODELS_DIR / artifact_filename

export_package = {
    'scaler': scaler,
    'model': val_ml_champion_model,
    'best_model_name': val_ml_champion_name,
    'feature_cols': feature_cols,
    'target_col': target_col,
    'target_unit': 'm3/m3',
    'horizon_hours': HORIZON,
    'agronomic_thresholds': {
        'field_capacity': FC,
        'wilting_point': WP,
        'critical_threshold': CRITICAL_THRESHOLD
    },
    'metadata': {
        'dataset_name': 'Gallipoli Sensor Time-Series',
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'val_ml_champion': val_ml_champion_name,
        'final_test_champion': final_test_champion,
        'persistence_test_mae': float(p_test_mae),
        'ml_test_mae': float(ml_test_mae),
        'mae_improvement_pct': float(mae_improvement_pct),
        'random_seed': SEED,
        'saved_at': time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    }
}

with open(artifact_path, 'wb') as f:
    pickle.dump(export_package, f)

artifact_size_mb = artifact_path.stat().st_size / (1024 * 1024)
print("Artifact Overwritten Successfully!")
print(f"  - Path: {artifact_path.resolve()}")
print(f"  - Size: {artifact_size_mb:.2f} MB")

# Reload Verification Check
with open(artifact_path, 'rb') as f:
    reloaded_dict = pickle.load(f)

reloaded_scaler = reloaded_dict['scaler']
reloaded_model = reloaded_dict['model']

X_sample = X_test.iloc[:10]
y_orig_sample = val_ml_champion_model.predict(scaler.transform(X_sample))
y_reload_sample = reloaded_model.predict(reloaded_scaler.transform(X_sample))

is_deterministic = np.allclose(y_orig_sample, y_reload_sample, atol=1e-6)
print(f"\nArtifact Reload Verification Check: Predictions Match 100%: {is_deterministic}")
assert is_deterministic, "CRITICAL FAILURE: Reloaded irrigation model predictions do not match!"
print("QUALITY GATE PASSED: Irrigation prediction artifact reloaded cleanly.")

Artifact Overwritten Successfully!
  - Path: D:\PROJECTS\AGRINEXUS-AI\Notebook\models\irrigation_prediction.pkl
  - Size: 0.00 MB



Artifact Reload Verification Check: Predictions Match 100%: True
QUALITY GATE PASSED: Irrigation prediction artifact reloaded cleanly.


In [8]:
# Section 9: Final Scientific Audit Table & Conclusions
final_audit_summary = [
    {"Metric / Aspect": "Dataset", "Audit Value": "Gallipoli Sensor Time-Series (SoilWaterContent.xlsx & porta1_meteo_piogge.xlsx)"},
    {"Metric / Aspect": "Dataset Size", "Audit Value": f"{len(df_clean):,} hourly observations ({len(X_train):,} train, {len(X_val):,} val, {len(X_test):,} test)"},
    {"Metric / Aspect": "Target Variable", "Audit Value": "3-Hour-Ahead Soil Water Content (SWC t+3h)"},
    {"Metric / Aspect": "Target Unit", "Audit Value": "m3/m3 (Volumetric Soil Water Content)"},
    {"Metric / Aspect": "Features", "Audit Value": f"{len(feature_cols)} features ({', '.join(feature_cols)})"},
    {"Metric / Aspect": "Split Strategy", "Audit Value": "Chronological Time-Series Split (70% Train / 15% Val / 15% Test)"},
    {"Metric / Aspect": "Causality Audit", "Audit Value": "PASS (Strict historical lags <= t; zero future interpolation or backward fill)"},
    {"Metric / Aspect": "Baseline Model", "Audit Value": "Persistence Baseline (SWC_t+3h = SWC_t)"},
    {"Metric / Aspect": "Candidate Models", "Audit Value": "Persistence, Ridge, LinearReg, Random Forest, Extra Trees, HistGB"},
    {"Metric / Aspect": "Validation ML Champion", "Audit Value": f"{val_ml_champion_name} (Val MAE = {best_val_mae:.6f} m3/m3)"},
    {"Metric / Aspect": "Final Test Champion", "Audit Value": f"{final_test_champion}"},
    {"Metric / Aspect": "Held-Out Test Metric", "Audit Value": f"Test MAE = {ml_test_mae:.6f} m3/m3, RMSE = {ml_test_rmse:.6f}, R2 = {ml_test_r2:.4f}"},
    {"Metric / Aspect": "Persistence Comparison", "Audit Value": f"Persistence Test MAE = {p_test_mae:.6f} m3/m3 | Improvement = {mae_improvement_pct:+.2f}%"},
    {"Metric / Aspect": "Decision Engine", "Audit Value": "Separated Agronomic Rules (Field Capacity = 0.320, Depletion Threshold = 0.230 m3/m3)"},
    {"Metric / Aspect": "Artifact Reload Result", "Audit Value": "PASS (Exact deterministic output match)"},
    {"Metric / Aspect": "Known Limitations", "Audit Value": "High SWC temporal persistence makes 3h horizon baseline strong; ML requires rain sensor spikes for gain"},
    {"Metric / Aspect": "Readiness Status", "Audit Value": readiness_status}
]

df_audit_summary = pd.DataFrame(final_audit_summary)
print("="*70)
print("FINAL MODEL AUDIT REPORT — SMART IRRIGATION FORECASTING")
print("="*70)
print(df_audit_summary.to_string(index=False))
print("="*70)

FINAL MODEL AUDIT REPORT — SMART IRRIGATION FORECASTING
       Metric / Aspect                                                                                             Audit Value
               Dataset                         Gallipoli Sensor Time-Series (SoilWaterContent.xlsx & porta1_meteo_piogge.xlsx)
          Dataset Size                                        14,588 hourly observations (10,211 train, 2,188 val, 2,189 test)
       Target Variable                                                              3-Hour-Ahead Soil Water Content (SWC t+3h)
           Target Unit                                                                   m3/m3 (Volumetric Soil Water Content)
              Features        7 features (SWC, SWC_lag1h, SWC_lag2h, SWC_lag3h, SWC_roll6h_mean, Rainfall_mm, Rain_roll6h_sum)
        Split Strategy                                        Chronological Time-Series Split (70% Train / 15% Val / 15% Test)
       Causality Audit                          PASS (S